# 第 45 章 · Swin 与 ViViT：只在小窗口里看，窗口每层错开半格；视频再加一条时间轴

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/45-swin-vivit.html) · 仓库里 `beyond-transformer/notes/lessons/45-swin-vivit.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 计算量 | 式 1、2 | `omega(56, 56, 96, 7)` | 全局 ≈ 20.0 亿，窗口 ≈ 1.45 亿；注意力项差 64 倍 |
| 第二步 分数格子 | | `score_cells(8, 4)` | 全局 4096，窗口 1024 |
| 第三步 哪个窗口 | 偏移 ⌊M/2⌋ | `window_id(r, c, M, shift)` | (3, 3) 和 (4, 4)：不错开时不同，错开 2 格时相同 |
| 第四步 合并 | 2 × 2 拼接 | `patch_merge(x)` | 8 × 8 × C → 4 × 4 × 4C |
| 第五步 视频 | tubelet | `video_tokens(32, 224, 224, 2, 16, 16)` | 3136 个；Model 1 ≈ 983 万格，Model 2 ≈ 61.5 万格 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `omega`**：Swin 式 1、2：全局 MSA 和窗口 W-MSA 的计算量（不算 softmax）。

In [ ]:
def omega(h, w, C, M):
    """Swin 式 1、2：全局 MSA 和窗口 W-MSA 的计算量（不算 softmax）。"""
    proj = 4 * h * w * C * C
    return {"proj": proj, "global_attn": 2 * (h * w) ** 2 * C, "window_attn": 2 * M * M * h * w * C}

**函数 `score_cells`**：side × side 的特征图：全局注意力和 M × M 窗口注意力各要算多少个分数格子。

In [ ]:
def score_cells(side, M):
    """side × side 的特征图：全局注意力和 M × M 窗口注意力各要算多少个分数格子。"""
    n = side * side
    windows = (side // M) ** 2
    return {"global": n * n, "window": windows * (M * M) ** 2}

**函数 `window_id`**：小块 (r, c) 落在哪个窗口：窗口往右下挪 shift，等于小块坐标减 shift。

In [ ]:
def window_id(r, c, M, shift=0):
    """小块 (r, c) 落在哪个窗口：窗口往右下挪 shift，等于小块坐标减 shift。"""
    r, c = r - shift, c - shift
    return (r // M, c // M)

**函数 `patch_merge`**：2 × 2 个相邻块首尾拼接：h × w × C → h/2 × w/2 × 4C。

In [ ]:
def patch_merge(x):
    """2 × 2 个相邻块首尾拼接：h × w × C → h/2 × w/2 × 4C。"""
    h, w, C = x.shape
    x = x.reshape(h // 2, 2, w // 2, 2, C)
    return x.transpose(0, 2, 1, 3, 4).reshape(h // 2, w // 2, 4 * C)

**函数 `video_tokens`**：ViViT tubelet：三个方向各切几段、一共多少 token、Model 1 和 Model 2 的分数格子数。

In [ ]:
def video_tokens(T, H, W, t, h, w):
    """ViViT tubelet：三个方向各切几段、一共多少 token、Model 1 和 Model 2 的分数格子数。"""
    nt, nh, nw = T // t, H // h, W // w
    n = nt * nh * nw
    return {"nt": nt, "nh": nh, "nw": nw, "tokens": n, "model1": n * n, "model2": nt * (nh * nw) ** 2 + nt**2}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 计算量：h = w = 56，C = 96，M = 7")
o = omega(56, 56, 96, 7)
print(o)
print(f"全局合计 {(o['proj'] + o['global_attn']) / 1e8:.2f} 亿，窗口合计 {(o['proj'] + o['window_attn']) / 1e8:.2f} 亿")
print("注意力那一项差", o["global_attn"] // o["window_attn"], "倍")

In [ ]:
print("\n第二步 分数格子：8 × 8，M = 4")
print(score_cells(8, 4))

In [ ]:
print("\n第三步 错开窗口：(3, 3) 和 (4, 4)")
for shift in (0, 2):
    print(f"shift = {shift}：", window_id(3, 3, 4, shift), window_id(4, 4, 4, shift))

In [ ]:
print("\n第四步 patch merging：8 × 8 × 3 → ?")
x = np.arange(8 * 8 * 3, dtype=float).reshape(8, 8, 3)
print(patch_merge(x).shape)

In [ ]:
print("\n第五步 视频：32 帧 224 × 224，tubelet 2 × 16 × 16")
print(video_tokens(32, 224, 224, 2, 16, 16))

## 3. 代码里才看得到的

- 讲义说"窗口往右下挪"，代码里写成"小块坐标减去 shift"，两者等价；负坐标用 Python 的 `//` 向下取整，碎窗口自然分到边上的编号里，正好对应原文说的 3 × 3 个窗口。
- patch merging 的拼接顺序（先行后列还是先列后行）各家实现不同，只要整个模型一致就行。

## 4. 自己改一改

- `omega(112, 112, 96, 7)`：图片边长翻倍，看两项各涨几倍。
- `window_id` 的 shift 改成 0、2、3，看 (3, 3) 和 (4, 4) 什么时候在同一个窗口。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 全校大会平方涨，分班开会线性长。
> 窗口里头开小会，位置偏置加分数上。
> 一层正切一层错，半格一挪信息传；碎窗卷起再遮挡。
> 两两拼块压两倍，四层金字塔像卷积。
> 视频切成小管子，先看帧内再看帧间。